# Day 5 — Evaluation (self-contained)

Through Day 4 I've been eyeballing answers. Today I turn "looks good" into a
number: generate a test set, run the agent on it, use LLM-as-judge to score.

**This notebook rebuilds the Day 4 agent from scratch** so it runs top-to-bottom
in one kernel with no dependency on Day 4. Run every cell in order.

In [63]:
# uv add minsearch sentence-transformers anthropic python-dotenv pandas numpy

In [64]:
import json, os, re, random, time
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import anthropic
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer

load_dotenv()
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5-20251001"
EMB_MODEL_NAME = "multi-qa-distilbert-cos-v1"                    # 768-dim
CACHE = f"embeddings.npy"     # model-named: can't collide
QUESTIONS_FILE = "eval_questions.json"
RESULTS_FILE = "eval_results.json"

## PART A — Rebuild the agent (from Day 4)
### A1. Load chunks + build both indexes

In [65]:
dbt_chunks = []
with open("chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        dbt_chunks.append(json.loads(line))
print(f"chunks: {len(dbt_chunks)}")

index = Index(text_fields=["chunk"], keyword_fields=["filename"])
index.fit(dbt_chunks)

embedding_model = SentenceTransformer(EMB_MODEL_NAME)

if os.path.exists(CACHE):
    embeddings = np.load(CACHE)
    print(f"loaded cached embeddings {embeddings.shape}")
else:
    texts = [c["chunk"] for c in dbt_chunks]
    embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True)
    np.save(CACHE, embeddings)
    print(f"encoded {embeddings.shape}")

# seatbelt: fail loudly BEFORE the confusing matrix error if dims disagree
assert embeddings.shape[0] == len(dbt_chunks), "embedding count != chunk count"
assert embeddings.shape[1] == embedding_model.encode("test").shape[0], "dim mismatch"

vindex = VectorSearch(keyword_fields=[])
vindex.fit(embeddings, dbt_chunks)
print("indexes ready")

chunks: 7910


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

loaded cached embeddings (7910, 768)
indexes ready


In [66]:
import os
print(os.getcwd())  

c:\Users\sanela.nikodinoska\aihero\project


In [67]:
import numpy as np
print(np.load("embeddings.npy").shape)   # want (7910, 768)

(7910, 768)


### A2. Search functions

In [68]:
def hybrid_search(query, num_results=5):
    lex = index.search(query, num_results=num_results)
    vec = vindex.search(embedding_model.encode(query), num_results=num_results)
    seen, merged = set(), []
    for pair in zip(lex, vec):                 # interleave: neither monopolizes top
        for r in pair:
            key = (r["filename"], r["chunk"][:50])
            if key not in seen:
                seen.add(key); merged.append(r)
    for r in lex + vec:                        # leftovers
        key = (r["filename"], r["chunk"][:50])
        if key not in seen:
            seen.add(key); merged.append(r)
    return merged[:num_results]


def text_search(query, num_results=5):
    results = hybrid_search(query, num_results=num_results)
    return [{"filename": r["filename"], "chunk": r["chunk"]} for r in results]


_hits = text_search("how do I schedule a dbt job")
print(f"search OK: {len(_hits)} hits, first {_hits[0]['filename']}")

search OK: 5 hits, first website/docs/guides/manual-install-qs.md


### A3. The agent (tool-use loop)

In [69]:
TOOLS = [{
    "name": "text_search",
    "description": (
        "Search the dbt documentation for relevant passages. Call this whenever "
        "you need factual information about dbt. You may call it MULTIPLE times "
        "with different queries to gather complete information. Returns "
        "documentation chunks with source filenames."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Search query"}},
        "required": ["query"],
    },
}]
TOOL_FUNCTIONS = {"text_search": text_search}

SYSTEM_PROMPT = (
    "You are a dbt documentation assistant. Answer using the text_search tool to "
    "ground every answer in the docs. Search as many times as needed. Cite the "
    "source filename for each claim. If the docs don't cover something, say so."
)


def run_agent(question, max_turns=6, verbose=False):
    messages = [{"role": "user", "content": question}]
    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=2048,
            system=SYSTEM_PROMPT, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"  [turn {turn}] search: {block.input.get('query')!r}")
                out = TOOL_FUNCTIONS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result", "tool_use_id": block.id,
                    "content": json.dumps(out),
                })
        messages.append({"role": "user", "content": tool_results})
    return "Stopped: reached max turns."


def ask_dbt(question):
    return run_agent(question, verbose=False)


print("ask_dbt ready:", callable(ask_dbt))

ask_dbt ready: True


## PART B — Evaluation
### B1. Generate the test question set

The LLM reads real chunks and writes questions each answers, giving a question
paired with a ground-truth source. Sampling only chunks >=500 chars.

In [74]:
def parse_json(text):
    """LLM JSON often has ```json fences or prose — extract the object/array."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    m = re.search(r"[\[{].*[\]}]", text, re.DOTALL)
    if m:
        text = m.group(0)
    return json.loads(text)


def generate_questions(chunk, n=2):
    prompt = f"""Based ONLY on the documentation below, write {n} distinct questions
a dbt user might ask that THIS text answers. Natural and specific.
Return ONLY a JSON list of strings, nothing else.

DOCUMENTATION:
{chunk['chunk'][:2000]}
"""
    resp = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    questions = parse_json(resp.content[0].text)
    return [{"question": q, "source": chunk["filename"]} for q in questions]


def build_question_set(chunks, n_chunks=25, per_chunk=2, seed=42):
    if os.path.exists(QUESTIONS_FILE):
        print("loading cached question set")
        return json.load(open(QUESTIONS_FILE, encoding="utf-8"))
    pool = [c for c in chunks if len(c["chunk"]) >= 500]
    random.seed(seed)
    sampled = random.sample(pool, min(n_chunks, len(pool)))
    questions = []
    for i, chunk in enumerate(sampled):
        try:
            questions.extend(generate_questions(chunk, per_chunk))
            print(f"  {i+1}/{len(sampled)} -> {len(questions)} questions")
        except Exception as e:
            print(f"  skip {i}: {e}")
        time.sleep(0.5)
    json.dump(questions, open(QUESTIONS_FILE, "w", encoding="utf-8"), indent=2)
    return questions


eval_questions = build_question_set(dbt_chunks, n_chunks=25, per_chunk=2)
print(f"\ntotal generated: {len(eval_questions)}")

  1/25 -> 2 questions
  2/25 -> 4 questions
  3/25 -> 6 questions
  4/25 -> 8 questions
  5/25 -> 10 questions
  6/25 -> 12 questions
  7/25 -> 14 questions
  8/25 -> 16 questions
  9/25 -> 18 questions
  10/25 -> 20 questions
  11/25 -> 22 questions
  12/25 -> 24 questions
  13/25 -> 26 questions
  14/25 -> 28 questions
  15/25 -> 30 questions
  16/25 -> 32 questions
  17/25 -> 34 questions
  18/25 -> 36 questions
  19/25 -> 38 questions
  20/25 -> 40 questions
  21/25 -> 42 questions
  22/25 -> 44 questions
  23/25 -> 46 questions
  24/25 -> 48 questions
  25/25 -> 50 questions

total generated: 50


### B2. Add known hard cases

In [75]:
hard_cases = [
    {"question": "How can I automate things in dbt?", "source": "MULTI (broad)"},
    {"question": "Can I use Wizard to create a new model out of an existing SQL query, based on source tables?",
     "source": "website/docs/docs/dbt-ai/wizard-use-cases.md"},
]
eval_questions = eval_questions + hard_cases
print(f"with hard cases: {len(eval_questions)}")

with hard cases: 52


### B3. Run the agent on every question (cached + checkpointed)

If you change the agent, delete `eval_results.json` first. This version also
auto-ignores cached ERROR rows so old failures don't stick.

In [ ]:
# import os
# for f in ["eval_results.json", "eval_questions.json"]:
#     if os.path.exists(f):
#         os.remove(f)
#         print("removed", f)

removed eval_results.json
removed eval_questions.json


In [76]:
def get_answers(questions):
    cache = {}
    if os.path.exists(RESULTS_FILE):
        cache = {r["question"]: r for r in json.load(open(RESULTS_FILE, encoding="utf-8"))}
    results = []
    for i, item in enumerate(questions):
        q = item["question"]
        if q in cache and "answer" in cache[q] and not cache[q]["answer"].startswith("ERROR"):
            results.append(cache[q]); continue
        try:
            answer = ask_dbt(q)
        except Exception as e:
            answer = f"ERROR: {e}"
        row = {**item, "answer": answer}
        results.append(row)
        print(f"  answered {i+1}/{len(questions)}")
        json.dump(results, open(RESULTS_FILE, "w", encoding="utf-8"), indent=2)
        time.sleep(0.5)
    return results


answered = get_answers(eval_questions)
errored = sum(r["answer"].startswith("ERROR") for r in answered)
print(f"\nanswered {len(answered)}, errored {errored}")
if errored:
    print("first error:", next(r['answer'] for r in answered if r['answer'].startswith('ERROR'))[:200])

  answered 1/52
  answered 2/52
  answered 3/52
  answered 4/52
  answered 5/52
  answered 6/52
  answered 7/52
  answered 8/52
  answered 9/52
  answered 10/52
  answered 11/52
  answered 12/52
  answered 13/52
  answered 14/52
  answered 15/52
  answered 16/52
  answered 17/52
  answered 18/52
  answered 19/52
  answered 20/52
  answered 21/52
  answered 22/52
  answered 23/52
  answered 24/52
  answered 25/52
  answered 26/52
  answered 27/52
  answered 28/52
  answered 29/52
  answered 30/52
  answered 31/52
  answered 32/52
  answered 33/52
  answered 34/52
  answered 35/52
  answered 36/52
  answered 37/52
  answered 38/52
  answered 39/52
  answered 40/52
  answered 41/52
  answered 42/52
  answered 43/52
  answered 44/52
  answered 45/52
  answered 46/52
  answered 47/52
  answered 48/52
  answered 49/52
  answered 50/52
  answered 51/52
  answered 52/52

answered 52, errored 0


### B4. LLM-as-judge

In [77]:
JUDGE_PROMPT = """You are evaluating a dbt documentation assistant's answer.

QUESTION: {question}

ANSWER: {answer}

Score 1-5 (5=best) on each:
- groundedness: factual, consistent with how dbt works, not made up?
- relevance: does it address the question?
- citation: does it cite source filenames?
- completeness: does it fully answer?

Return ONLY this JSON:
{{"groundedness": N, "relevance": N, "citation": N, "completeness": N, "comment": "one sentence"}}
"""


def judge_answer(question, answer):
    resp = client.messages.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "user",
                   "content": JUDGE_PROMPT.format(question=question, answer=answer)}],
    )
    return parse_json(resp.content[0].text)


def run_judge(answered):
    scored, failures = [], 0
    for i, row in enumerate(answered):
        if row["answer"].startswith("ERROR"):
            continue
        try:
            scores = judge_answer(row["question"], row["answer"])
            scored.append({**row, **scores})
        except Exception as e:
            failures += 1
            print(f"  judge parse failed on {i}: {e}")
        time.sleep(0.5)
    print(f"\njudged {len(scored)}, judge-failures {failures}")
    return scored


scored = run_judge(answered)


judged 52, judge-failures 0


### B5. Aggregate — the number that replaces "looks good"

In [78]:
df = pd.DataFrame(scored)
criteria = ["groundedness", "relevance", "citation", "completeness"]

print("=== mean scores (1-5) ===")
print(df[criteria].mean().round(2).to_string())
print(f"\noverall mean: {df[criteria].mean().mean():.2f}")

print("\n=== weakest answers (lowest groundedness) ===")
for _, r in df.nsmallest(3, "groundedness").iterrows():
    print(f"\nQ: {r['question'][:70]}")
    print(f"   g={r['groundedness']} r={r['relevance']} c={r['citation']} comp={r['completeness']}")
    print(f"   judge: {r.get('comment', '')}")

=== mean scores (1-5) ===
groundedness    3.37
relevance       4.04
citation        3.67
completeness    3.48

overall mean: 3.64

=== weakest answers (lowest groundedness) ===

Q: What are the key structural changes in the new Semantic Layer YAML spe
   g=1 r=1 c=2 comp=1
   judge: The answer appears to contain fabricated information about dbt Core v1.12 changes that cannot be verified against actual dbt documentation, including non-existent file paths and features that don't align with dbt's actual Semantic Layer evolution.

Q: How should I update my metric definitions when upgrading to dbt Core v
   g=1 r=1 c=2 comp=1
   judge: This answer appears to be entirely fabricated—dbt Core v1.12 does not exist (current version is v1.8 as of early 2024), the `dbt-autofix` tool is not real, the file paths cited don't match actual dbt documentation structure, and the semantic layer spec changes described do not correspond to any actual dbt release.

Q: What are the trusted adapter options avai

### B6. Hard cases vs. generated

Meta-test: did known-hard questions score below generated single-hop ones? If
yes, the eval discriminates. If everything scores 5/5, it measures nothing.

In [79]:
hard_qs = {c["question"] for c in hard_cases}
hard_df = df[df["question"].isin(hard_qs)]
easy_df = df[~df["question"].isin(hard_qs)]

print("hard cases mean:", round(hard_df[criteria].mean().mean(), 2))
print("generated mean :", round(easy_df[criteria].mean().mean(), 2))
hard_df[["question"] + criteria] if len(hard_df) else "no hard cases scored"

hard cases mean: 3.75
generated mean : 3.64


,question,groundedness,relevance,citation,completeness
50,How can I automate things in dbt?,4,5,5,4
51,Can I use Wizard to create a new model out of ...,2,4,3,3


## Day 5 findings

*(Fill in with your actual numbers.)*

1. **Overall score: __/5** — the single trackable number. Every future change
   gets measured against it instead of eyeballed.
2. **Weakest criterion is likely citation** — agents answer correctly but forget
   to name the source. If so, fix the system prompt, not retrieval.
3. **Hard vs. generated:** if hard cases scored lower, the eval discriminates.
4. **The judge is a proxy, not truth** — it can be fooled by fluent-but-wrong
   answers. Spot-check a few by hand. Use it for trends/regressions.
5. **Lesson learned the hard way:** spreading the agent across notebooks caused
   repeated "not defined" and dimension errors. Making Day 5 self-contained (and
   naming the embedding cache after the model) killed both bug classes. This is
   exactly what Day 6's refactor into shared `.py` modules formalizes.

**Next (Day 6):** pull search + agent into `.py` modules, wrap `ask_dbt` in a
Streamlit app.

In [83]:
# see the Wizard answer and why the judge dinged it
row = df[df["question"].str.contains("Wizard")].iloc[0]
print("ANSWER:\n", row["answer"])
print("\nJUDGE COMMENT:", row["comment"])

# watch what the agent actually searched for
print("\n--- trace ---")
run_agent("Can I use Wizard to create a new model out of an existing SQL query, based on source tables?", verbose=True)

ANSWER:
 Based on the dbt documentation, here are the ways you can configure which AI model and reasoning effort level Wizard uses:

## Configuration Methods

There are **three ways** to configure the AI model and reasoning effort, in order of precedence:

### 1. **CLI Flags at Invocation** (highest priority)
Use the `-m` or `-c` flags when running Wizard from the command line to override settings temporarily.

### 2. **In-Session Model Picker** (`/model` command)
While in a Wizard session, use the `/model` slash command to choose your model and reasoning effort level interactively through the text-based user interface (TUI). According to the documentation, this command lets you "Choose what model and reasoning effort to use" ([_wizard-slash-commands-generated.md](website/docs/docs/dbt-ai/_wizard-slash-commands-generated.md)).

### 3. **Configuration Files** (lower priority)
Wizard uses two configuration files:

- **`~/.dbt/wizard/config.toml`** — Sets the default AI model globally acr

'Based on my search results, I can provide you with information about using Wizard to create models:\n\n## Yes, You Can Create Models with Wizard\n\nAccording to the dbt documentation, **dbt Wizard can be used to create new models from SQL queries based on source tables**. Here\'s what you can do:\n\n### How to Build a New Model\n\nIn the **Studio IDE**, according to the documentation (_snippets/_wizard-ide.md):\n\nYou can write natural language prompts to build new models. An example prompt format would be:\n\n> "Create a model called `fct_daily_revenue` that joins `stg_orders` and `stg_payments`, aggregates revenue by day, and materializes as a table."\n\n### Steps to Use Wizard for Model Creation\n\nAccording to the same source (_snippets/_wizard-ide.md), here\'s the workflow:\n\n1. Open your dbt project in the Studio IDE and click **Wizard** in the command palette\n2. Use plain text to describe what you want to build\n3. Include:\n   - **Scope**: Which models or area you\'re workin

This is a genuinely interesting finding, and it's not the agent being dumb. It's lexical ambiguity — "model" has two meanings in dbt-land, and the agent's self-chosen query landed on the wrong sense. Your Day 3 manual query worked because you phrased it with disambiguating words ("create a new model out of an existing SQL query, based on source tables") that steered vector search to the right chunk. The agent compressed that into a shorter query and lost the disambiguation. Confirm with a trace: 

In [84]:
run_agent("Can I use Wizard to create a new model out of an existing SQL query, based on source tables?", verbose=True)

  [turn 0] search: 'Wizard create model SQL query source tables'
  [turn 0] search: 'Wizard SQL model creation'
  [turn 1] search: 'Wizard refactor existing SQL query into model'
  [turn 1] search: 'Wizard convert raw SQL to dbt model'
  [turn 2] search: 'Wizard build model from existing query source'
  [turn 2] search: 'Wizard create model based on'


"Based on the dbt documentation, **yes, you can use Wizard to create a new model based on an existing SQL query and source tables**. Here's what the documentation shows:\n\n## Building a New Model with Wizard\n\nAccording to the dbt Wizard use cases documentation (`wizard-use-cases.md`), Wizard supports **building a new model** as one of its core use cases. The example demonstrates:\n\n**Example prompt:**\n```text\nCreate a model called `fct_monthly_revenue` that joins `stg_orders` and `stg_payments`,\ngroups by `month` and `customer_id`, and materializes as a table. Add `not_null` tests\nto the primary key and a unique test on the grain.\n```\n\n**How Wizard handles this:**\n1. Reads your existing source models/staging tables from your project index to understand available columns\n2. Generates the new SQL model file with the specified join and aggregation logic\n3. Creates matching YAML configuration with tests\n4. Shows you a diff of both files for review before saving\n\n## Key Cap

Two possible scenarios: 
The right chunk was retrieved but the agent didn't use it. All four searches returned results, the fct_monthly_revenue example may well have been in there, but the agent synthesized its answer from the wrong chunks — it latched onto the AI-model-config pages instead of the build-a-model example. That's a synthesis failure, not retrieval.
The right chunk didn't rank even with good queries. Despite proper phrasing, hybrid search returned the config/slash-command pages above the use-cases example across all four searches.
Find out which — check whether the answer chunk was actually in what the agent saw:


In [85]:
# did any of the agent's 4 searches surface the answer chunk?
for q in ["Wizard create model SQL query source tables",
          "Wizard SQL model generation",
          "Wizard build new model existing query",
          "dbt Wizard create model from source"]:
    hits = text_search(q, num_results=5)
    found = any("fct_monthly_revenue" in h["chunk"] for h in hits)
    files = [h["filename"].split("/")[-1] for h in hits]
    print(f"{'✓' if found else '✗'} {q[:35]:37} -> {files[:3]}")

✗ Wizard create model SQL query sourc   -> ['wizard-how-it-works.md', 'wizard-7-semantic-layer.md', 'wizard-2-understand-project.md']
✓ Wizard SQL model generation           -> ['wizard-how-it-works.md', 'wizard-7-semantic-layer.md', 'wizard-2-understand-project.md']
✗ Wizard build new model existing que   -> ['wizard-how-it-works.md', 'wizard-prompts.md', 'wizard-use-cases.md']
✗ dbt Wizard create model from source   -> ['wizard-how-it-works.md', '_generate-metadata.md', 'wizard-2-understand-project.md']


The root cause is one document dominating retrieval. wizard-how-it-works.md is keyword-dense about Wizard generally, so it ranks top for every wizard query, crowding out the specific wizard-use-cases.md chunk that actually answers the question. The signal (one good chunk) got drowned by repetition of a mediocre one.

This is genuinely different from the two failures we theorized, and it's a well-known RAG problem with known fixes:

Fix 1 — deduplicate by filename across the agent's searches. Right now if wizard-how-it-works.md returns in all 4 searches, the agent sees it 4× and weights it heavily. Collapse duplicates so each source counts once:

In [88]:
def text_search(query, num_results=8):
    results = hybrid_search(query, num_results=num_results)
    seen_files, deduped = set(), []
    for r in results:
        if r["filename"] not in seen_files:
            seen_files.add(r["filename"])
            deduped.append(r)
    return [{"filename": r["filename"], "chunk": r["chunk"]} for r in deduped]

results = text_search("Wizard create model SQL query source tables", num_results=5)
results

[{'filename': 'website/docs/docs/dbt-ai/wizard-how-it-works.md',
  'chunk': 'How dbt Wizard works\nUnderstand the dbt Wizard agent loop, tools, project context, and approval workflow.\n\n## Related docs\n\n- [<Constant name="wizard" /> overview](/docs/platform/wizard-overview)\n- [<Constant name="wizard" /> in the dbt platform](/docs/platform/wizard-platform)\n- [Use <Constant name="wizard" /> locally](/docs/dbt-ai/wizard-quickstart)\n- [<Constant name="wizard" /> command reference](/docs/dbt-ai/wizard-cli-reference)\n- [How to use dbt Wizard in your dbt project](/best-practices/how-to-use-wizard/wizard-1-intro) for recommended workflows'},
 {'filename': 'website/docs/best-practices/how-to-use-wizard/wizard-7-semantic-layer.md',
  'chunk': 'Building Semantic Layer definitions with dbt Wizard\nUse dbt Wizard to plan, write, and validate Semantic Layer entities, dimensions, metrics, and saved queries.\n\n## Ask Wizard to plan the definitions\n\nGive <Constant name="wizard" /> a specific 

The answer lives under ## Build a new model — a self-contained 985-character section. If Day 2 had chunked on headers instead of sliding windows, this would be one distinct chunk with a clear title, instead of smeared across 11 near-identical overlapping fragments.

So the real story, fully traced:

The agent searched well (4 good queries).
The answer is in the corpus.
But Day 2's sliding-window chunking cut wizard-use-cases.md into 18 overlapping fragments, 11 of which look nearly identical to search. The one with the answer (chunk 8) can't be distinguished from its look-alike siblings, so it doesn't reliably surface.
The agent synthesized from the chunks that did surface — the keyword-dense overview — and answered the wrong sense of "model."

The fix is header-based chunking, which your Day 2 split_markdown_by_level already does — but your saved chunks.jsonl was built with the sliding window as the primary splitter.

Day 5's eval caught that the Wizard question scored groundedness 2. Tracing it: the agent's searches were well-formed and the answer existed in the corpus, but Day 2's sliding-window chunking had split the source file into 18 overlapping near-duplicate fragments, smearing the answer across look-alikes that search couldn't distinguish. Header-based chunking isolates the answer into a single clean ## Build a new model section. This closes the loop: a Day 5 evaluation finding points directly back to a Day 2 chunking decision — which is exactly why evaluation exists.